# Data loading and preprocessing

## Dataset loading

In [40]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# For type hinting, which is good practice
from torch.utils.data import Dataset
from typing import Tuple

def load_fashion_data() -> Tuple[Dataset, Dataset, Dataset]:
    """
    Loads the Fashion-MNIST dataset and splits the data into train, val, test.

    This function loads the Fashion-MNIST training and test sets from
    torchvision. It then splits the original training set into a new,
    smaller training set and a validation set (90/10 ratio).

    Returns:
        Tuple[Dataset, Dataset, Dataset]: A tuple containing the training,
        validation, and test datasets as torch.utils.data.Dataset objects.
    """
    # Use a basic transform to convert images to tensors.
    # More complex transforms will be applied later in the custom Dataset class.
    # converts a PIL Image or numpy.ndarray to tensor.
    transform = transforms.ToTensor()

    # 1. Load the full original training data
    full_train_dataset = datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # 2. Load the test data
    test_dataset = datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # 3. Calculate split sizes
    dataset_size = len(full_train_dataset)
    train_size = int(0.9 * dataset_size)
    val_size = dataset_size - train_size

    # 4. Split the full training dataset into training and validation subsets
    # Using a fixed generator seed for reproducibility of the split
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    return train_dataset, val_dataset, test_dataset

# # Example of how to call the function and verify the output
# if __name__ == '__main__':
#     train_data, val_data, test_data = load_fashion_data()

#     print(f"Fashion-MNIST Dataset Summary:")
#     print(f"  - Total Original Training Images: {len(train_data) + len(val_data)}")
#     print(f"  - Split Training Set Size:      {len(train_data)}")
#     print(f"  - Split Validation Set Size:    {len(val_data)}")
#     print(f"  - Test Set Size:                {len(test_data)}")

## Custom Dataset Class

In [41]:
import torch
from torch.utils.data import Dataset, Subset
from torchvision import transforms
from typing import Tuple, Any

def calculate_mean_std(dataset: Dataset) -> Tuple[Tuple[float, ...], Tuple[float, ...]]:
    """
    Calculate the mean and standard deviation of a dataset.
    
    Args:
        dataset: A PyTorch dataset containing images
        
    Returns:
        Tuple containing mean and std as tuples
    """
    # Stack all images into a single tensor
    # We'll iterate through the dataset and collect all pixel values
    all_pixels = []
    
    for img, _ in dataset:
        # img is already a tensor from ToTensor() transform
        all_pixels.append(img)
    
    # Stack all images: shape will be [N, C, H, W]
    all_pixels = torch.stack(all_pixels)
    
    # Calculate mean and std across all dimensions except the channel dimension
    # Compute per-channel statistics
    mean = all_pixels.mean(dim=[0, 2, 3])  # Average over batch, height, width
    std = all_pixels.std(dim=[0, 2, 3])    # Std dev over batch, height, width
    
    return tuple(mean.tolist()), tuple(std.tolist())

# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

In [42]:
# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

In [43]:
class FashionMNISTDataset(Dataset):
    """
    Custom Dataset wrapper for Fashion-MNIST that returns:
    - Normalized image tensor
    - Class label
    - Ink target (normalized pixel intensity)
    
    Supports optional data augmentation for training.
    """
    
    def __init__(
        self, 
        base_dataset: Dataset,
        mean: Tuple[float, ...],
        std: Tuple[float, ...],
        augment: bool = False
    ):
        """
        Initialize the custom Fashion-MNIST dataset.
        
        Args:
            base_dataset: The underlying Fashion-MNIST dataset (can be a Subset)
            mean: Mean values for normalization (per channel)
            std: Standard deviation values for normalization (per channel)
            augment: Whether to apply data augmentation (for training only)
        """
        self.base_dataset = base_dataset
        self.mean = mean
        self.std = std
        self.augment = augment
        
        # Create normalization transform
        self.normalize = transforms.Normalize(mean=mean, std=std)
        
        # Create augmentation transforms (light augmentations)
        if self.augment:
            self.augmentation = transforms.Compose([
                # Random crop: crop to 28x28 from a 32x32 padded image
                # This creates slight translations
                transforms.RandomCrop(28, padding=4),
                
                # Small random rotation (up to 10 degrees)
                transforms.RandomRotation(degrees=10),
                
                # Random horizontal flip (50% chance)
                # Note: This is reasonable for most Fashion-MNIST items
                transforms.RandomHorizontalFlip(p=0.5),
            ])
        
    def __len__(self) -> int:
        """Return the total number of samples in the dataset."""
        return len(self.base_dataset)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, float]:
        """
        Get a single item from the dataset.
        
        Args:
            idx: Index of the item to retrieve
            
        Returns:
            Tuple containing:
                - image: Normalized (and optionally augmented) image tensor of shape [C, H, W]
                - class_label: Integer label in range [0, 9]
                - ink_target: Normalized pixel intensity (average pixel value)
        """
        # Get the original image and label from the base dataset
        image, class_label = self.base_dataset[idx]
        
        # Calculate ink target BEFORE any transformations
        # Ink is the average pixel value (mean intensity) of the original image
        # image is already a tensor from ToTensor() with values in [0, 1]
        ink_target = image.mean().item()
        
        # Apply augmentation if enabled (training set only)
        if self.augment:
            image = self.augmentation(image)
        
        # Normalize the image using the dataset mean and std
        normalized_image = self.normalize(image)
        
        return normalized_image, class_label, ink_target

## Data Augmentation

We apply light augmentations to the training set only to improve generalization:
- **Random Crop**: Crop 28x28 from a 32x32 padded image (creates small translations)
- **Random Rotation**: Small rotations up to ±10 degrees
- **Random Horizontal Flip**: 50% chance of horizontal flip

Validation and test sets remain unaltered for fair evaluation.

In [44]:
# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

# print("\n" + "="*60)

# # Create custom dataset instances
# # Training set WITH augmentation
# train_custom = FashionMNISTDataset(
#     train_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=True  # Enable augmentation for training
# )

# # Validation and test sets WITHOUT augmentation
# val_custom = FashionMNISTDataset(
#     val_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=False  # No augmentation for validation
# )

# test_custom = FashionMNISTDataset(
#     test_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=False  # No augmentation for test
# )

# print("\nCustom datasets created successfully!")
# print(f"Training set size: {len(train_custom)} (with augmentation)")
# print(f"Validation set size: {len(val_custom)} (no augmentation)")
# print(f"Test set size: {len(test_custom)} (no augmentation)")

# # Test the dataset by getting one sample
# sample_image, sample_label, sample_ink = train_custom[0]
# print(f"\nSample from training dataset:")
# print(f"  Image shape: {sample_image.shape}")
# print(f"  Image dtype: {sample_image.dtype}")
# print(f"  Class label: {sample_label}")
# print(f"  Ink target: {sample_ink:.4f}")
# print(f"  Image min value: {sample_image.min():.4f}")
# print(f"  Image max value: {sample_image.max():.4f}")

In [45]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Visualize the effect of augmentation
# # Get the same sample multiple times to see different augmentations
# fig, axes = plt.subplots(2, 5, figsize=(15, 6))
# fig.suptitle('Effect of Data Augmentation on Training Set', fontsize=16, fontweight='bold')

# # Get one sample index
# sample_idx = 0

# # Row 1: Original (no augmentation)
# # axes[0, 0].set_ylabel('Without Augmentation', fontsize=12, fontweight='bold')
# # Create a non-augmented dataset temporarily for comparison

# train_no_aug = FashionMNISTDataset(
#     train_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=False
# )
# img, label, ink = train_no_aug[sample_idx]

# # Denormalize for visualization
# img_denorm = img * FASHION_MNIST_STD[0] + FASHION_MNIST_MEAN[0]
# img_denorm = img_denorm.squeeze().numpy()

# for col in range(5):
    
#     axes[0, col].imshow(img_denorm, cmap='gray')
#     axes[0, col].axis('off')
#     if col == 0:
#         axes[0, col].set_title(f'Label: {label}\nInk: {ink:.3f}', fontsize=10)
#     else:
#         axes[0, col].set_title('(Same Image)', fontsize=10)

# # Row 2: With augmentation (different each time)
# # axes[1, 0].set_ylabel('With Augmentation', fontsize=12, fontweight='bold')
# for col in range(5):
#     img, label, ink = train_custom[sample_idx]
    
#     # Denormalize for visualization
#     img_denorm = img * FASHION_MNIST_STD[0] + FASHION_MNIST_MEAN[0]
#     img_denorm = img_denorm.squeeze().numpy()
    
#     axes[1, col].imshow(img_denorm, cmap='gray')
#     axes[1, col].axis('off')
#     if col == 0:
#         axes[1, col].set_title(f'Label: {label}\nInk: {ink:.3f}', fontsize=10)
#     else:
#         axes[1, col].set_title('(Random Aug)', fontsize=10)

# plt.tight_layout()
# plt.show()

# print("Note: Each time you access an augmented sample, different random transformations are applied!")
# print("This helps the model generalize better by seeing variations of the same image.")

In [46]:
# # Create custom dataset instances
# train_custom = FashionMNISTDataset(train_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)
# val_custom = FashionMNISTDataset(val_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)
# test_custom = FashionMNISTDataset(test_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)

# print("Custom datasets created successfully!")
# print(f"Training set size: {len(train_custom)}")
# print(f"Validation set size: {len(val_custom)}")
# print(f"Test set size: {len(test_custom)}")

# # Test the dataset by getting one sample
# sample_image, sample_label, sample_ink = train_custom[0]
# print(f"\nSample from dataset:")
# print(f"  Image shape: {sample_image.shape}")
# print(f"  Image dtype: {sample_image.dtype}")
# print(f"  Class label: {sample_label}")
# print(f"  Ink target: {sample_ink:.4f}")
# print(f"  Image min value: {sample_image.min():.4f}")
# print(f"  Image max value: {sample_image.max():.4f}")

# Model Implementation: Multi-Task CNN

## Architecture, Forward Pass

In [47]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiTaskCNN(nn.Module):
    """
    Multi-Task CNN for Fashion-MNIST classification and ink regression.
    
    Architecture:
    - Shared convolutional backbone with 3 Conv-BatchNorm-ReLU-Pool blocks
    - Two separate heads:
        1. Classification head: outputs logits [B, 10] for 10 classes
        2. Regression head: outputs scalar values [B, 1] for ink prediction
    """
    
    def __init__(self, dropout_rate: float = 0.5):
        """
        Initialize the Multi-Task CNN.
        
        Args:
            dropout_rate: Dropout probability for regularization (default: 0.5)
        """
        super(MultiTaskCNN, self).__init__()
        
        # =====================================================================
        # SHARED CONVOLUTIONAL BACKBONE
        # =====================================================================
        
        # Block 1: Input [B, 1, 28, 28] -> Output [B, 32, 14, 14]
        self.conv1 = nn.Conv2d(
            in_channels=1,      # Grayscale input
            out_channels=32,    # 32 feature maps
            kernel_size=3,      # 3x3 convolution
            padding=1           # Keep spatial dimensions
        )
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14
        
        # Block 2: Input [B, 32, 14, 14] -> Output [B, 64, 7, 7]
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7
        
        # Block 3: Input [B, 64, 7, 7] -> Output [B, 128, 3, 3]
        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1
        )
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 7x7 -> 3x3
        
        # Dropout for regularization
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # Calculate the size of flattened features
        # After 3 pooling layers: 28 -> 14 -> 7 -> 3
        # Feature map size: 128 * 3 * 3 = 1152
        self.feature_size = 128 * 3 * 3
        
        # =====================================================================
        # CLASSIFICATION HEAD (for class labels)
        # =====================================================================
        self.fc_class1 = nn.Linear(self.feature_size, 256)
        self.bn_class1 = nn.BatchNorm1d(256)
        self.fc_class2 = nn.Linear(256, 10)  # 10 classes for Fashion-MNIST
        
        # =====================================================================
        # REGRESSION HEAD (for ink prediction)
        # =====================================================================
        self.fc_regress1 = nn.Linear(self.feature_size, 128)
        self.bn_regress1 = nn.BatchNorm1d(128)
        self.fc_regress2 = nn.Linear(128, 1)  # Single scalar output
        
    def forward_shared(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the shared convolutional backbone.
        
        Args:
            x: Input tensor of shape [B, 1, 28, 28]
            
        Returns:
            Flattened features of shape [B, feature_size]
        """
        # Block 1: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        # Block 2: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        # Block 3: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        
        # Flatten the feature maps
        x = x.view(x.size(0), -1)  # [B, 128*3*3] = [B, 1152]
        
        # Apply dropout
        x = self.dropout(x)
        
        return x
    
    def forward(
        self, 
        x: torch.Tensor, 
        return_features: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor] | tuple[torch.Tensor, torch.Tensor, dict]:
        """
        Forward pass through the entire network.
        
        Args:
            x: Input tensor of shape [B, 1, 28, 28]
            return_features: If True, also return intermediate feature maps for visualization
            
        Returns:
            If return_features=False:
                Tuple containing:
                    - class_logits: Classification logits of shape [B, 10]
                    - ink_pred: Ink predictions of shape [B, 1]
            
            If return_features=True:
                Tuple containing:
                    - class_logits: Classification logits of shape [B, 10]
                    - ink_pred: Ink predictions of shape [B, 1]
                    - features: Dictionary with intermediate feature maps:
                        'conv1': Output after first conv block [B, 32, 14, 14]
                        'conv2': Output after second conv block [B, 64, 7, 7]
                        'conv3': Output after third conv block [B, 128, 3, 3]
                        'shared_features': Flattened features [B, 1152]
        """
        # Dictionary to store intermediate features if requested
        features = {} if return_features else None
        
        # ===== Shared Convolutional Backbone =====
        
        # Block 1: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        if return_features:
            features['conv1'] = x.clone()  # [B, 32, 14, 14]
        
        # Block 2: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        if return_features:
            features['conv2'] = x.clone()  # [B, 64, 7, 7]
        
        # Block 3: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        if return_features:
            features['conv3'] = x.clone()  # [B, 128, 3, 3]
        
        # Flatten the feature maps
        x = x.view(x.size(0), -1)  # [B, 128*3*3] = [B, 1152]
        
        # Apply dropout
        shared_features = self.dropout(x)
        if return_features:
            features['shared_features'] = shared_features.clone()  # [B, 1152]
        
        # ===== Classification Head =====
        x_class = self.fc_class1(shared_features)
        x_class = self.bn_class1(x_class)
        x_class = F.relu(x_class)
        x_class = self.dropout(x_class)
        class_logits = self.fc_class2(x_class)  # [B, 10]
        
        # ===== Regression Head =====
        x_regress = self.fc_regress1(shared_features)
        x_regress = self.bn_regress1(x_regress)
        x_regress = F.relu(x_regress)
        x_regress = self.dropout(x_regress)
        ink_pred = self.fc_regress2(x_regress)  # [B, 1]
        
        # Return with or without features
        if return_features:
            return class_logits, ink_pred, features
        else:
            return class_logits, ink_pred
    
    def get_num_parameters(self) -> int:
        """
        Calculate the total number of trainable parameters in the model.
        
        Returns:
            Total number of trainable parameters
        """
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [48]:
# # Create an instance of the model
# model = MultiTaskCNN(dropout_rate=0.5)

# # Display model architecture
# print("="*70)
# print("Multi-Task CNN Architecture")
# print("="*70)
# print(model)
# print("="*70)

# # Count parameters
# total_params = model.get_num_parameters()
# print(f"\nTotal Trainable Parameters: {total_params:,}")

# # Test with a dummy input
# dummy_input = torch.randn(4, 1, 28, 28)  # Batch of 4 images
# print(f"\nTest Input Shape: {dummy_input.shape}")

# # Forward pass
# class_logits, ink_pred = model(dummy_input)

# print(f"\nOutput Shapes:")
# print(f"  Classification Logits: {class_logits.shape} (expected: [4, 10])")
# print(f"  Ink Predictions:       {ink_pred.shape} (expected: [4, 1])")

# print("\n✅ Model created successfully!")
# print("✅ Both heads (classification and regression) are working correctly!")

In [49]:
# # Detailed architecture summary
# print("\n" + "="*70)
# print("DETAILED ARCHITECTURE BREAKDOWN")
# print("="*70)

# print("\n📦 SHARED CONVOLUTIONAL BACKBONE:")
# print("-" * 70)
# print("Block 1:")
# print("  Conv2d(1 -> 32, kernel=3x3, padding=1)  [28x28 -> 28x28]")
# print("  BatchNorm2d(32)")
# print("  ReLU()")
# print("  MaxPool2d(2x2)                           [28x28 -> 14x14]")

# print("\nBlock 2:")
# print("  Conv2d(32 -> 64, kernel=3x3, padding=1)  [14x14 -> 14x14]")
# print("  BatchNorm2d(64)")
# print("  ReLU()")
# print("  MaxPool2d(2x2)                            [14x14 -> 7x7]")

# print("\nBlock 3:")
# print("  Conv2d(64 -> 128, kernel=3x3, padding=1) [7x7 -> 7x7]")
# print("  BatchNorm2d(128)")
# print("  ReLU()")
# print("  MaxPool2d(2x2)                            [7x7 -> 3x3]")

# print("\nDropout(p=0.5)")
# print("Flatten: [B, 128, 3, 3] -> [B, 1152]")

# print("\n" + "-" * 70)
# print("🎯 CLASSIFICATION HEAD (Class Labels):")
# print("-" * 70)
# print("  Linear(1152 -> 256)")
# print("  BatchNorm1d(256)")
# print("  ReLU()")
# print("  Dropout(p=0.5)")
# print("  Linear(256 -> 10)  [B, 10] logits")

# print("\n" + "-" * 70)
# print("📊 REGRESSION HEAD (Ink Prediction):")
# print("-" * 70)
# print("  Linear(1152 -> 128)")
# print("  BatchNorm1d(128)")
# print("  ReLU()")
# print("  Dropout(p=0.5)")
# print("  Linear(128 -> 1)  [B, 1] scalar output")

# print("\n" + "="*70)

# # Calculate parameters for each component
# def count_params(module):
#     return sum(p.numel() for p in module.parameters())

# backbone_params = (
#     count_params(model.conv1) + count_params(model.bn1) +
#     count_params(model.conv2) + count_params(model.bn2) +
#     count_params(model.conv3) + count_params(model.bn3)
# )

# class_head_params = (
#     count_params(model.fc_class1) + count_params(model.bn_class1) +
#     count_params(model.fc_class2)
# )

# regress_head_params = (
#     count_params(model.fc_regress1) + count_params(model.bn_regress1) +
#     count_params(model.fc_regress2)
# )

# print("\n📊 PARAMETER BREAKDOWN:")
# print("-" * 70)
# print(f"  Shared Backbone:      {backbone_params:>10,} parameters")
# print(f"  Classification Head:  {class_head_params:>10,} parameters")
# print(f"  Regression Head:      {regress_head_params:>10,} parameters")
# print("-" * 70)
# print(f"  TOTAL:                {total_params:>10,} parameters")
# print("="*70)

### Architecture Summary

#### ✅ Key Features:

1. **Shared Convolutional Backbone** (3 blocks):
   - Each block follows: **Conv2d → BatchNorm2d → ReLU → MaxPool2d**
   - Progressive channel increase: 1 → 32 → 64 → 128
   - Spatial dimension reduction: 28×28 → 14×14 → 7×7 → 3×3
   - **Dropout** applied after backbone for regularization

2. **Classification Head**:
   - Input: Flattened features [B, 1152]
   - Architecture: FC(1152→256) → BatchNorm → ReLU → Dropout → FC(256→10)
   - Output: Logits [B, 10] for 10 Fashion-MNIST classes

3. **Regression Head**:
   - Input: Flattened features [B, 1152]
   - Architecture: FC(1152→128) → BatchNorm → ReLU → Dropout → FC(128→1)
   - Output: Scalar values [B, 1] for ink prediction

### 📊 Model Statistics:
- **Total Parameters**: 539,339
- **Shared Backbone**: 93,120 parameters (17.3%)
- **Classification Head**: 298,250 parameters (55.3%)
- **Regression Head**: 147,969 parameters (27.4%)

The model efficiently shares feature extraction while maintaining separate task-specific heads!

### Testing Feature Visualization

In [50]:
# # Test forward pass with feature visualization
# print("="*70)
# print("TESTING FORWARD PASS WITH FEATURE VISUALIZATION")
# print("="*70)

# # Create a dummy input
# test_input = torch.randn(2, 1, 28, 28)
# print(f"\nInput Shape: {test_input.shape}")

# # Test 1: Normal forward pass (without features)
# print("\n" + "-"*70)
# print("Test 1: Normal Forward Pass (return_features=False)")
# print("-"*70)
# class_logits, ink_pred = model(test_input, return_features=False)
# print(f"✓ Classification Logits: {class_logits.shape}")
# print(f"✓ Ink Predictions:       {ink_pred.shape}")

# # Test 2: Forward pass with feature visualization
# print("\n" + "-"*70)
# print("Test 2: Forward Pass with Features (return_features=True)")
# print("-"*70)
# class_logits, ink_pred, features = model(test_input, return_features=True)

# print(f"✓ Classification Logits: {class_logits.shape}")
# print(f"✓ Ink Predictions:       {ink_pred.shape}")
# print(f"\n✓ Intermediate Feature Maps:")
# for key, value in features.items():
#     print(f"    {key:20s}: {str(value.shape):25s}")

# print("\n" + "="*70)
# print("✅ Forward pass working correctly!")
# print("✅ Feature visualization enabled!")
# print("="*70)

In [51]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Visualize intermediate feature maps
# print("\n" + "="*70)
# print("VISUALIZING INTERMEDIATE FEATURE MAPS")
# print("="*70)

# # Get a single sample for visualization
# single_input = test_input[0:1]  # Take first sample, keep batch dimension

# # Set model to evaluation mode (required for BatchNorm with batch_size=1)
# model.eval()

# # Forward pass with features
# with torch.no_grad():  # No need for gradients during visualization
#     class_logits, ink_pred, features = model(single_input, return_features=True)

# # Create visualization
# fig, axes = plt.subplots(3, 8, figsize=(16, 6))
# fig.suptitle('Intermediate Feature Maps at Each Convolutional Block', 
#              fontsize=14, fontweight='bold')

# # Visualize first 8 feature maps from each conv block
# blocks = ['conv1', 'conv2', 'conv3']
# block_names = ['Block 1 (32 channels)', 'Block 2 (64 channels)', 'Block 3 (128 channels)']

# for row, (block, block_name) in enumerate(zip(blocks, block_names)):
#     feature_map = features[block][0]  # Remove batch dimension [C, H, W]
    
#     # Plot first 8 channels
#     for col in range(8):
#         if col < feature_map.shape[0]:  # Check if channel exists
#             channel = feature_map[col].cpu().numpy()
#             axes[row, col].imshow(channel, cmap='viridis')
#             axes[row, col].axis('off')
#             if col == 0:
#                 axes[row, col].set_title(f'{block_name}\nChannel {col}', 
#                                         fontsize=8, loc='left')
#             else:
#                 axes[row, col].set_title(f'Ch {col}', fontsize=8)
#         else:
#             axes[row, col].axis('off')

# plt.tight_layout()
# plt.show()

# print("\n📊 Feature Map Details:")
# print(f"  Conv Block 1: {features['conv1'].shape} -> Shows low-level features (edges, textures)")
# print(f"  Conv Block 2: {features['conv2'].shape} -> Shows mid-level features (patterns)")
# print(f"  Conv Block 3: {features['conv3'].shape} -> Shows high-level features (object parts)")
# print(f"  Shared Features: {features['shared_features'].shape} -> Flattened for classification/regression")

# print("\n✅ Feature maps extracted and visualized successfully!")

### Forward Pass Implementation Summary

#### ✅ Implemented Requirements:

1. **Pass inputs through shared convolutional layers** ✓
   - Input flows through 3 Conv-BatchNorm-ReLU-Pool blocks
   - Features are extracted hierarchically from low to high level
   - Shared backbone ensures both tasks benefit from same feature representations

2. **Output both classification logits and regression predictions** ✓
   - Classification head outputs logits `[B, 10]` for 10 Fashion-MNIST classes
   - Regression head outputs scalar values `[B, 1]` for ink prediction
   - Both outputs returned as a tuple: `(class_logits, ink_pred)`

3. **Optionally return intermediate feature maps for visualization** ✓
   - `return_features=False` (default): Returns only `(class_logits, ink_pred)`
   - `return_features=True`: Returns `(class_logits, ink_pred, features_dict)`
   - Features dictionary contains:
     - `'conv1'`: Features after Block 1 `[B, 32, 14, 14]`
     - `'conv2'`: Features after Block 2 `[B, 64, 7, 7]`
     - `'conv3'`: Features after Block 3 `[B, 128, 3, 3]`
     - `'shared_features'`: Flattened features `[B, 1152]`

#### 🎯 Key Benefits:

- **Flexibility**: Can be used for both training (without features) and analysis (with features)
- **Efficiency**: No performance overhead when features are not needed
- **Debugging**: Intermediate features help understand what the network learns
- **Visualization**: Can inspect feature maps at different abstraction levels